In [1]:
import os
os.makedirs("agents", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("static", exist_ok=True)

In [2]:
%%writefile agents/__init__.py

Writing agents/__init__.py


In [3]:
%%writefile agents/intent_agent.py
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

class IntentAgent:
    def __init__(self):
        self.pipeline = Pipeline([
            ("tfidf", TfidfVectorizer(stop_words="english", max_features=5000)),
            ("clf", LogisticRegression(max_iter=1000)),
        ])
        self.is_trained = False

    def train(self, csv_path, text_col="Ticket Description", label_col="Ticket Type"):
        df = pd.read_csv(csv_path).dropna(subset=[text_col, label_col])
        X, y = df[text_col], df[label_col]
        can_stratify = y.value_counts().min() >= 2
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y if can_stratify else None
        )
        self.pipeline.fit(X_train, y_train)
        self.is_trained = True
        print(f"[IntentAgent] Trained on {len(X_train)} tickets.")

    def predict(self, ticket_description):
        return self.pipeline.predict([ticket_description])[0]

Writing agents/intent_agent.py


In [4]:
%%writefile agents/retrieval_agent.py
import json
import numpy as np
from sentence_transformers import SentenceTransformer

class RetrievalAgent:
    def __init__(self, kb_path, model_name="all-MiniLM-L6-v2"):
        with open(kb_path) as f:
            self.entries = json.load(f)["knowledge_base"]
        print(f"[RetrievalAgent] Loading '{model_name}'...")
        self.model = SentenceTransformer(model_name)
        texts = [f"{e['subject']}: {e['sample_query']}" for e in self.entries]
        self.embeddings = self.model.encode(texts, normalize_embeddings=True)
        print(f"[RetrievalAgent] Indexed {len(self.entries)} entries.")

    def retrieve(self, query, top_k=1):
        query_emb = self.model.encode([query], normalize_embeddings=True)[0]
        sims = self.embeddings @ query_emb
        top_idx = np.argsort(sims)[::-1][:top_k]
        return [{"subject": self.entries[i]["subject"],
                  "answer": self.entries[i]["answer"],
                  "similarity": float(sims[i])} for i in top_idx]

Writing agents/retrieval_agent.py


In [5]:
%%writefile agents/response_agent.py
import os
from groq import Groq

class ResponseAgent:
    def __init__(self, mode="llm", model="llama-3.3-70b-versatile"):
        self.mode = mode
        self.model = model
        if mode == "llm":
            self.client = Groq(api_key=os.environ["GROQ_API_KEY"])

    def draft_reply(self, customer_name, product, ticket_description, kb_answer):
        if self.mode == "template":
            return (f"Hi {customer_name},\n\nThanks for reaching out about your {product}. "
                    f"{kb_answer}\n\nLet us know if this resolves the issue or if you need "
                    f"further help.\n\nBest,\nSupport Team")
        prompt = (f"You are a customer support agent. Write a short, warm, professional reply.\n\n"
                  f"Customer name: {customer_name}\nProduct: {product}\n"
                  f"Customer's issue: {ticket_description}\nReference solution: {kb_answer}\n\n"
                  f"Write only the reply text, 3-5 sentences.")
        response = self.client.chat.completions.create(
            model=self.model, max_tokens=300,
            messages=[{"role": "user", "content": prompt}])
        return response.choices[0].message.content

Writing agents/response_agent.py


In [6]:
%%writefile agents/escalation_agent.py
class EscalationAgent:
    def __init__(self, similarity_threshold=0.45, always_escalate_priorities=("Critical", "High")):
        self.similarity_threshold = similarity_threshold
        self.always_escalate_priorities = set(always_escalate_priorities)

    def decide(self, similarity, ticket_priority=None):
        if ticket_priority in self.always_escalate_priorities:
            return {"action": "escalate",
                    "reason": f"Priority '{ticket_priority}' — routed to a human regardless of confidence."}
        if similarity < self.similarity_threshold:
            return {"action": "escalate",
                    "reason": f"Retrieval confidence too low ({similarity:.2f})."}
        return {"action": "auto_resolve",
                "reason": f"High confidence match ({similarity:.2f}), priority '{ticket_priority}' is safe."}

Writing agents/escalation_agent.py


In [7]:
%%writefile data/knowledge_base.json
{
  "knowledge_base": [
    {"subject": "Delivery problem", "sample_query": "My {product_purchased} hasn't arrived and it's past the estimated delivery date.",
     "answer": "We're sorry for the delay. Please check your order tracking link for the latest carrier update. If the package shows no movement for more than 3 business days, we'll issue a replacement or full refund — just confirm your order number and we'll process it right away."},
    {"subject": "Software bug", "sample_query": "My {product_purchased} keeps freezing or crashing when I use a specific feature.",
     "answer": "This is often fixed by updating to the latest firmware/app version, then restarting the device. If the issue persists after updating, try clearing the app's cache or performing a soft reset. Let us know your current software version if the problem continues."},
    {"subject": "Battery life", "sample_query": "The battery on my {product_purchased} drains much faster than it used to.",
     "answer": "Reduced battery life is usually caused by background processes or an aging battery. Try disabling unused features (Bluetooth, location, always-on display) and check for a firmware update. If the battery still degrades quickly after 30+ charge cycles, it may need to be serviced under warranty."},
    {"subject": "Product recommendation", "sample_query": "I'm not sure which accessory or model works best with my {product_purchased}.",
     "answer": "Happy to help you choose. Compatibility depends on your device's model number and intended use case — could you share those details? In the meantime, our officially certified accessories page lists everything guaranteed to work with your device."},
    {"subject": "Refund request", "sample_query": "I'd like a refund for my {product_purchased} because it isn't working as expected.",
     "answer": "We're sorry to hear that. Refunds are available within 30 days of purchase with proof of purchase. Please share your order number and reason for the return, and we'll send you a prepaid return label along with refund timing details."},
    {"subject": "Product setup", "sample_query": "I can't get my {product_purchased} set up correctly out of the box.",
     "answer": "Start by making sure the device is fully charged and running the latest firmware before setup. Follow the in-app or in-box quick start guide step by step — most setup failures happen when a step is skipped. Let us know exactly which step it fails on and we'll walk you through it."},
    {"subject": "Data loss", "sample_query": "I lost important data or files stored on my {product_purchased}.",
     "answer": "First, avoid using the device further to prevent overwriting recoverable data. Check if the device syncs to a cloud backup, since files are often recoverable from there. If not, we can advise on manufacturer-approved data recovery options."},
    {"subject": "Installation support", "sample_query": "I'm having trouble installing software or drivers for my {product_purchased}.",
     "answer": "Make sure you're downloading the installer directly from the official product page, not a third-party source. Disable antivirus temporarily during install if it's blocking the process, and confirm your operating system meets the minimum requirements listed on the product page."},
    {"subject": "Account access", "sample_query": "I can't log into my account associated with my {product_purchased}.",
     "answer": "Try the 'Forgot password' reset link first — most access issues are resolved this way. If you're not receiving the reset email, check your spam folder or confirm the email on file. If you're still locked out, we can manually verify your identity and restore access."},
    {"subject": "Peripheral compatibility", "sample_query": "An accessory or peripheral won't connect properly to my {product_purchased}.",
     "answer": "Confirm the accessory is listed as officially compatible with your device model. Re-pair the connection from scratch (unpair, restart both devices, then pair again), and make sure both devices are running current firmware, as compatibility issues are often resolved by updates."},
    {"subject": "Display issue", "sample_query": "The screen on my {product_purchased} is flickering, discolored, or unresponsive.",
     "answer": "Try a full restart first, as this resolves most temporary display glitches. If the issue persists, check for a pending firmware update. Persistent flickering or dead pixels after a restart usually indicates a hardware fault and may be covered under warranty."},
    {"subject": "Hardware issue", "sample_query": "My {product_purchased} is making unusual noises or physically malfunctioning.",
     "answer": "Please stop using the device if you notice unusual heat, noise, or smell, for safety. Confirm whether the issue happens consistently or intermittently, and let us know your purchase date — if it's within the warranty period, we'll arrange a repair or replacement."},
    {"subject": "Cancellation request", "sample_query": "I want to cancel my order or subscription for {product_purchased}.",
     "answer": "We can process that for you. If the order hasn't shipped yet, cancellation is immediate and free. If it has already shipped, you're welcome to refuse delivery or return it once received for a full refund under our standard return policy."},
    {"subject": "Payment issue", "sample_query": "I was charged incorrectly or my payment failed for {product_purchased}.",
     "answer": "Sorry for the trouble. Duplicate or incorrect charges are usually reversed automatically within 3-5 business days. If you don't see a correction by then, share the transaction ID and charge amount and we'll investigate and issue a manual refund if needed."},
    {"subject": "Network problem", "sample_query": "My {product_purchased} won't connect to Wi-Fi or keeps losing connection.",
     "answer": "Restart both the device and your router first. Make sure the device is within range and connected to a 2.4GHz network if it doesn't support 5GHz. If the connection still drops intermittently, try forgetting the network and reconnecting from scratch."},
    {"subject": "Product compatibility", "sample_query": "I'm not sure if {product_purchased} works with my other devices or software.",
     "answer": "Compatibility depends on your device's model and software version. Please share both, and we'll confirm whether it's supported or suggest a workaround. Our compatibility chart on the product support page also covers most common pairings."}
  ]
}

Writing data/knowledge_base.json


In [8]:
from google.colab import files
uploaded = files.upload()  # select your Kaggle CSV

import shutil
csv_filename = list(uploaded.keys())[0]
shutil.move(csv_filename, "data/tickets.csv")
print("Saved to data/tickets.csv")

Saving customer_support_tickets.csv to customer_support_tickets.csv
Saved to data/tickets.csv


In [9]:
%%writefile app.py
import os
from flask import Flask, request, jsonify, send_from_directory
from agents.intent_agent import IntentAgent
from agents.retrieval_agent import RetrievalAgent
from agents.response_agent import ResponseAgent
from agents.escalation_agent import EscalationAgent

app = Flask(__name__, static_folder="static")

print("Initializing agents...")
intent_agent = IntentAgent()
intent_agent.train("data/tickets.csv")

retrieval_agent = RetrievalAgent("data/knowledge_base.json")
response_agent = ResponseAgent(mode="llm")
escalation_agent = EscalationAgent(similarity_threshold=0.45)
print("All agents ready.")


@app.route("/")
def index():
    return send_from_directory("static", "index.html")


@app.route("/api/process", methods=["POST"])
def process():
    data = request.get_json()
    customer_name = data.get("customer_name", "Customer")
    product = data.get("product", "your product")
    ticket_description = data.get("ticket_description", "")
    ticket_priority = data.get("ticket_priority", "Medium")

    if not ticket_description.strip():
        return jsonify({"error": "Ticket description is required."}), 400

    match = retrieval_agent.retrieve(ticket_description, top_k=1)[0]
    predicted_intent = intent_agent.predict(ticket_description)
    draft_reply = response_agent.draft_reply(
        customer_name, product, ticket_description, match["answer"]
    )
    escalation = escalation_agent.decide(match["similarity"], ticket_priority)

    return jsonify({
        "matched_subject": match["subject"],
        "retrieval_similarity": round(match["similarity"], 3),
        "predicted_intent": predicted_intent,
        "draft_reply": draft_reply,
        "escalation_action": escalation["action"],
        "escalation_reason": escalation["reason"],
    })


if __name__ == "__main__":
    port = int(os.environ.get("PORT", 7860))
    app.run(host="0.0.0.0", port=port)

Writing app.py


In [10]:
%%writefile static/index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Support Copilot</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,400;9..144,600&family=Inter:wght@400;500;600;700&display=swap" rel="stylesheet">
<style>
  :root {
    --paper: #EEF1F4;
    --card: #FFFFFF;
    --ink: #1B1F3B;
    --ink-soft: #565B7A;
    --indigo: #3A4B8C;
    --indigo-dark: #2C3A6E;
    --teal: #2F9E68;
    --teal-bg: #E4F5EC;
    --rust: #C1502E;
    --rust-bg: #FBEAE4;
    --line: #DADEE6;
    --radius: 10px;
  }
  * { box-sizing: border-box; }
  body { margin: 0; background: var(--paper); color: var(--ink); font-family: 'Inter', sans-serif; line-height: 1.5; }
  header { padding: 40px 32px 24px; max-width: 1100px; margin: 0 auto; }
  header h1 { font-family: 'Fraunces', serif; font-weight: 600; font-size: 34px; margin: 0 0 6px; letter-spacing: -0.01em; }
  header p { color: var(--ink-soft); margin: 0; max-width: 60ch; font-size: 15px; }
  main { max-width: 1100px; margin: 0 auto; padding: 8px 32px 64px; display: grid; grid-template-columns: 1fr 1fr; gap: 24px; }
  @media (max-width: 860px) { main { grid-template-columns: 1fr; } }
  .card { background: var(--card); border: 1px solid var(--line); border-radius: var(--radius); padding: 28px; }
  .card h2 { font-size: 15px; font-weight: 600; margin: 0 0 20px; color: var(--ink); }
  label { display: block; font-size: 13px; font-weight: 500; color: var(--ink-soft); margin-bottom: 6px; }
  .field { margin-bottom: 18px; }
  input[type="text"], textarea, select { width: 100%; padding: 10px 12px; border: 1px solid var(--line); border-radius: 8px; font-family: inherit; font-size: 14px; color: var(--ink); background: #FCFCFD; transition: border-color 0.15s ease; }
  input[type="text"]:focus, textarea:focus, select:focus { outline: none; border-color: var(--indigo); box-shadow: 0 0 0 3px rgba(58, 75, 140, 0.12); }
  textarea { resize: vertical; min-height: 96px; }
  .row { display: grid; grid-template-columns: 1fr 1fr; gap: 14px; }
  button { width: 100%; background: var(--indigo); color: white; border: none; border-radius: 8px; padding: 12px 16px; font-family: inherit; font-size: 14px; font-weight: 600; cursor: pointer; transition: background 0.15s ease; }
  button:hover { background: var(--indigo-dark); }
  button:disabled { background: #B7BEDA; cursor: not-allowed; }
  .trail { display: flex; flex-direction: column; gap: 0; }
  .step { display: grid; grid-template-columns: 28px 1fr; gap: 14px; opacity: 0.35; transition: opacity 0.25s ease; }
  .step.active { opacity: 1; }
  .step-marker { display: flex; flex-direction: column; align-items: center; }
  .step-dot { width: 22px; height: 22px; border-radius: 50%; background: var(--line); display: flex; align-items: center; justify-content: center; font-size: 11px; font-weight: 700; color: white; flex-shrink: 0; }
  .step.active .step-dot { background: var(--indigo); }
  .step.done .step-dot { background: var(--teal); }
  .step-line { width: 2px; flex: 1; background: var(--line); margin-top: 2px; }
  .step-body { padding-bottom: 22px; }
  .step-label { font-size: 13px; font-weight: 600; margin-bottom: 4px; }
  .step-detail { font-size: 13px; color: var(--ink-soft); }
  .badge { display: inline-flex; align-items: center; gap: 8px; padding: 10px 16px; border-radius: 8px; font-size: 14px; font-weight: 600; margin-bottom: 4px; }
  .badge.resolve { background: var(--teal-bg); color: var(--teal); }
  .badge.escalate { background: var(--rust-bg); color: var(--rust); }
  .reply-box { margin-top: 18px; padding: 16px; background: #FAFAFB; border: 1px solid var(--line); border-radius: 8px; font-size: 14px; white-space: pre-wrap; color: var(--ink); }
  .copy-btn { background: none; border: 1px solid var(--line); color: var(--ink-soft); font-size: 12px; font-weight: 500; padding: 6px 12px; width: auto; margin-top: 10px; }
  .copy-btn:hover { background: #F2F3F6; color: var(--ink); }
  .empty-state { color: var(--ink-soft); font-size: 14px; text-align: center; padding: 60px 20px; }
  .error-text { color: var(--rust); font-size: 13px; margin-top: 10px; }
</style>
</head>
<body>

<header>
  <h1>Support Copilot</h1>
  <p>Describe a customer's issue and watch four agents work through it together — matching it to a known solution, drafting a reply, and deciding whether it's safe to send or needs a person.</p>
</header>

<main>
  <section class="card">
    <h2>Ticket details</h2>
    <form id="ticket-form">
      <div class="row">
        <div class="field">
          <label for="customer_name">Customer name</label>
          <input type="text" id="customer_name" value="Jane Doe">
        </div>
        <div class="field">
          <label for="product">Product</label>
          <input type="text" id="product" value="iPhone">
        </div>
      </div>
      <div class="field">
        <label for="ticket_description">What's the issue?</label>
        <textarea id="ticket_description" placeholder="e.g. My laptop screen keeps flickering and sometimes goes black."></textarea>
      </div>
      <div class="field">
        <label for="ticket_priority">Priority</label>
        <select id="ticket_priority">
          <option>Low</option>
          <option selected>Medium</option>
          <option>High</option>
          <option>Critical</option>
        </select>
      </div>
      <button type="submit" id="submit-btn">Run through the agents</button>
      <div class="error-text" id="error-text"></div>
    </form>
  </section>

  <section class="card">
    <h2>Agent trail</h2>
    <div id="result-panel">
      <div class="empty-state" id="empty-state">Submit a ticket to see how each agent handles it.</div>
    </div>
  </section>
</main>

<script>
const form = document.getElementById('ticket-form');
const submitBtn = document.getElementById('submit-btn');
const resultPanel = document.getElementById('result-panel');
const errorText = document.getElementById('error-text');

function renderTrail() {
  resultPanel.innerHTML = `
    <div class="trail">
      <div class="step active" id="step-intent">
        <div class="step-marker"><div class="step-dot">1</div><div class="step-line"></div></div>
        <div class="step-body">
          <div class="step-label">Reading the ticket</div>
          <div class="step-detail" id="detail-intent">Classifying intent…</div>
        </div>
      </div>
      <div class="step" id="step-retrieval">
        <div class="step-marker"><div class="step-dot">2</div><div class="step-line"></div></div>
        <div class="step-body">
          <div class="step-label">Finding a matching solution</div>
          <div class="step-detail" id="detail-retrieval">Waiting…</div>
        </div>
      </div>
      <div class="step" id="step-reply">
        <div class="step-marker"><div class="step-dot">3</div><div class="step-line"></div></div>
        <div class="step-body">
          <div class="step-label">Drafting a reply</div>
          <div class="step-detail" id="detail-reply">Waiting…</div>
        </div>
      </div>
      <div class="step" id="step-escalation">
        <div class="step-marker"><div class="step-dot">4</div></div>
        <div class="step-body">
          <div class="step-label">Deciding what happens next</div>
          <div class="step-detail" id="detail-escalation">Waiting…</div>
        </div>
      </div>
    </div>
  `;
}

function copyReply(text) {
  navigator.clipboard.writeText(text);
}

form.addEventListener('submit', async (e) => {
  e.preventDefault();
  errorText.textContent = '';

  const ticket_description = document.getElementById('ticket_description').value.trim();
  if (!ticket_description) {
    errorText.textContent = 'Describe the issue before running the agents.';
    return;
  }

  submitBtn.disabled = true;
  submitBtn.textContent = 'Running…';
  renderTrail();

  const payload = {
    customer_name: document.getElementById('customer_name').value || 'Customer',
    product: document.getElementById('product').value || 'your product',
    ticket_description,
    ticket_priority: document.getElementById('ticket_priority').value,
  };

  try {
    const res = await fetch('/api/process', {
      method: 'POST',
      headers: { 'Content-Type': 'application/json' },
      body: JSON.stringify(payload),
    });

    if (!res.ok) {
      const err = await res.json();
      throw new Error(err.error || 'Something went wrong.');
    }

    const data = await res.json();

    document.getElementById('step-intent').classList.add('done');
    document.getElementById('detail-intent').textContent = `Secondary signal: ${data.predicted_intent}`;

    document.getElementById('step-retrieval').classList.add('active', 'done');
    document.getElementById('detail-retrieval').textContent =
      `Matched "${data.matched_subject}" (confidence ${data.retrieval_similarity})`;

    document.getElementById('step-reply').classList.add('active', 'done');
    document.getElementById('detail-reply').textContent = 'Reply drafted below.';

    document.getElementById('step-escalation').classList.add('active', 'done');
    const isResolve = data.escalation_action === 'auto_resolve';
    document.getElementById('detail-escalation').innerHTML = `
      <span class="badge ${isResolve ? 'resolve' : 'escalate'}">
        ${isResolve ? 'Auto-resolved' : 'Escalated to a human'}
      </span>
      <div>${data.escalation_reason}</div>
    `;

    const replyBox = document.createElement('div');
    replyBox.className = 'reply-box';
    replyBox.textContent = data.draft_reply;
    document.getElementById('step-reply').querySelector('.step-body').appendChild(replyBox);

    const copyBtn = document.createElement('button');
    copyBtn.className = 'copy-btn';
    copyBtn.textContent = 'Copy reply';
    copyBtn.type = 'button';
    copyBtn.onclick = () => {
      copyReply(data.draft_reply);
      copyBtn.textContent = 'Copied';
      setTimeout(() => (copyBtn.textContent = 'Copy reply'), 1500);
    };
    document.getElementById('step-reply').querySelector('.step-body').appendChild(copyBtn);

  } catch (err) {
    errorText.textContent = err.message;
    resultPanel.innerHTML = `<div class="empty-state" id="empty-state">Submit a ticket to see how each agent handles it.</div>`;
  } finally {
    submitBtn.disabled = false;
    submitBtn.textContent = 'Run through the agents';
  }
});
</script>

</body>
</html>

Writing static/index.html


In [11]:
%%writefile requirements.txt
flask
pandas
numpy
scikit-learn
sentence-transformers
groq

Writing requirements.txt


In [12]:
%%writefile Procfile
web: python app.py

Writing Procfile


In [13]:
%%writefile .gitignore
__pycache__/
*.pyc
.env

Writing .gitignore


In [14]:
import shutil
shutil.make_archive("support-agent-app", "zip", ".", base_dir=".")
from google.colab import files
files.download("support-agent-app.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>